## 환경 준비

아래 셀은 이 노트북에 필요한 Python 패키지가 설치되어 있는지 확인하고,
없으면 자동으로 설치한다. 이미 설치되어 있으면 빠르게 스킵된다.
터미널에서 미리 `uv sync`를 했다면 이 셀은 아무것도 설치하지 않는다.


In [ ]:
# === 의존성 자동 설치 (이미 설치되어 있으면 빠르게 스킵됨) ===
import subprocess, sys

_IMPORT_NAME_OVERRIDES = {
    "scikit-learn": "sklearn",
    "python-dateutil": "dateutil",
    "beautifulsoup4": "bs4",
}


def _ensure_packages(*packages):
    """누락된 패키지만 설치. 이미 있으면 스킵."""
    missing = []
    for pkg in packages:
        name = pkg.split(">=")[0].split("==")[0].split("[")[0]
        import_name = _IMPORT_NAME_OVERRIDES.get(name, name.replace("-", "_"))
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("All packages already installed ✓")

_ensure_packages(
    "pandas", "numpy", "matplotlib", "scikit-learn",
    "pydantic", "python-dateutil", "rich", "tqdm",
)


# RAG 기초(RAG Basics)

이 노트북은 가장 단순한 형태의 검색 증강 생성(RAG, Retrieval-Augmented Generation) 파이프라인을 해부한다. 먼저 문서를 읽고, 문장을 청크(chunk)로 자르고, TF-IDF 기반 검색기로 관련 청크를 찾은 뒤, 그 근거를 이어 붙여 답변을 만든다. 구조 자체는 단순하지만, 이후 02번의 agentic workflow가 왜 필요한지 이해하는 출발점이 된다.

## 학습 목표
- RAG가 전통적인 검색 엔진과 무엇이 같고 무엇이 다른지 설명할 수 있다.
- 문서 적재(load) → 청킹(chunking) → 검색(retrieval) → 답변 생성의 흐름을 코드 수준에서 따라갈 수 있다.
- TF-IDF 기반 baseline과 `build_demo_index(backend='faiss')`로 여는 dense retrieval 경로의 차이를 말할 수 있다.
- baseline이 왜 계획(planning), 도구(tool use), 근거 검증(grounding verification), abstain 로직 없이 취약한지 설명할 수 있다.


## 개념 설명

RAG는 모델이 스스로 모든 사실을 기억한다고 가정하지 않고, 질문이 들어올 때 외부 문서를 먼저 검색한 뒤 그 검색 결과를 바탕으로 답하게 만드는 구조다. 전통적인 검색 엔진이 "관련 문서를 보여주는 것"에 집중한다면, RAG는 "관련 문서를 찾아서 답변까지 조합하는 것"까지 범위가 넓다.

여기서 baseline이라는 표현은 의도적으로 중요하다. 지금 보는 구현은 빠르고 설명하기 쉬우며 재현성이 높지만, reasoning step을 분리하지 않고, 툴 호출도 없고, 답변이 실제 근거에 충분히 기반했는지 별도로 점검하지 않는다. 그래서 01번은 "잘 되는 예시"보다 "어디까지가 단순 RAG의 한계인가"를 배우는 튜토리얼로 읽는 편이 좋다.

**목적**
- 이후 셀들의 구현을 보기 전에 전체 파이프라인의 좌표를 잡는다.
- 검색과 생성이 하나의 모델 안에서 일어나는 것이 아니라, 데이터 흐름으로 분리된다는 점을 먼저 이해한다.

**핵심 로직**
- 문서를 로드한다.
- 문서를 여러 문장 단위 청크로 자른다.
- 각 청크를 벡터화해 질의와의 유사도를 계산한다.
- 상위 청크를 근거로 답변 문자열을 만든다.

**주요 파라미터/변수**
- `documents`: 원문 문서 목록
- `chunks`: 검색 가능한 최소 단위
- `top_k`: 검색 결과로 몇 개의 청크를 가져올지 결정하는 값

**결과 해석 가이드**
- 이 노트북의 표와 점수는 모두 "어떤 근거가 검색되었는가"를 중심으로 읽으면 된다.
- 검색 결과가 맞아도 답변 문장이 어색할 수 있고, 반대로 문장은 자연스러워도 검색이 틀리면 시스템 전체는 실패다.

**💡 면접 포인트**
- "Baseline RAG는 빠르고 설명 가능하지만, planning·tool use·verification이 없어 복합 질문과 불충분 근거 상황에 약하다"고 정리하면 좋다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## RAG란 무엇인가

아래 셀은 먼저 원문 문서를 읽어 `doc_id`, `source`, `text` 구조로 메모리에 올린다. RAG에서 이 단계가 중요한 이유는, 이후의 모든 청크와 citation이 결국 이 원문 문서 식별자에 기대기 때문이다. 문서 로딩이 느슨하면 나중에 어떤 답변이 어떤 원문에서 왔는지 설명하기 어려워진다.

**목적**
- 검색 전에 원문 문서가 어떤 단위로 메모리에 올라오는지 확인한다.

**핵심 로직**
- `load_documents()`는 기본 raw 디렉터리를 찾고, demo 폴더가 있으면 그 하위 문서를 우선 사용한다.
- 재귀 탐색이 꺼져 있으면 현재 디렉터리의 파일만 읽고, 켜져 있으면 하위 폴더까지 내려간다.
- 파일 확장자가 `.md`, `.txt`인 것만 적재한다.

**주요 파라미터/변수**
- `raw_dir`: 별도 말이 없으면 `data/raw/demo`를 사용한다.
- `recursive=False`: 이 baseline notebook은 중첩 폴더보다 단순한 corpus를 먼저 다룬다.
- `document_frame`: 로드 결과를 표로 읽기 쉽게 만든 DataFrame이다.

**실제 소스 코드: load_documents() — src/ingestion.py**
```python
def load_documents(raw_dir: Path | None = None, recursive: bool = False) -> list[dict[str, str]]:
    paths = get_paths()
    source_dir = raw_dir or paths.raw_dir
    if raw_dir is None and (source_dir / "demo").exists():
        source_dir = source_dir / "demo"
    documents: list[dict[str, str]] = []

    iterator = source_dir.rglob("*") if recursive else source_dir.iterdir()
    for path in sorted(iterator):
        if not path.is_file():
            continue
        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        if path.parent == source_dir:
            source = path.name
            doc_id = path.stem
        else:
            relative_path = path.relative_to(source_dir)
            source = str(relative_path)
            doc_id = "__".join(relative_path.with_suffix("").parts)
        documents.append(
            {
                "doc_id": doc_id,
                "source": source,
                "text": path.read_text(),
            }
        )

    return documents
```

**코드 읽기 포인트**
- `source_dir = raw_dir or paths.raw_dir`: 외부에서 raw 디렉터리를 주지 않으면 프로젝트 기본 경로를 쓴다.
- `if raw_dir is None and (source_dir / "demo").exists()`: demo 프로필이 있으면 교육용 문서를 자동으로 선택한다.
- `iterator = source_dir.rglob("*") if recursive else source_dir.iterdir()`: 재귀 탐색 여부를 여기서 결정한다.
- `source`는 citation에 쓰일 사람이 읽는 이름이고, `doc_id`는 내부 식별자다.
- 반환 구조를 dict list로 단순하게 유지해서 notebooks, evaluator, workflow가 모두 같은 형태를 재사용한다.

**결과 해석 가이드**
- `characters`가 너무 짧은 문서는 검색 실험에서 잡음(noise)이 되기 쉽다.
- `source`는 나중에 citation 표시에 그대로 쓰이므로 사람이 알아보기 쉬운지 함께 확인하면 좋다.


In [ ]:
import pandas as pd

from src.ingestion import load_documents

documents = load_documents()
document_frame = pd.DataFrame(
    [
        {'doc_id': item['doc_id'], 'source': item['source'], 'characters': len(item['text'])}
        for item in documents
    ]
)
document_frame

## 문서 청킹(chunking) 이해하기

문서 전체를 한 번에 검색하면 긴 문서 안의 중요한 문장이 묻혀 버린다. 그래서 RAG는 보통 문서를 더 작은 단위로 자른 뒤 검색한다. 이 저장소는 문장 기반 청킹을 쓰는데, 이유는 학습용 프로젝트에서 "어느 문장이 왜 검색되었는지"를 사람이 읽기 쉽기 때문이다.

문장 기반 청킹과 고정 길이 청킹의 트레이드오프도 꼭 기억하자. 문장 기반은 해석이 쉬운 대신 문장 수가 들쭉날쭉해 길이 편차가 크고, 고정 길이는 길이 통제가 쉬운 대신 문장 경계가 잘릴 수 있다.

**목적**
- 검색 단위를 왜 문서 전체가 아니라 청크로 만드는지 이해한다.

**핵심 로직**
- `ingest_documents()`는 각 문서를 순회하며 `chunk_document()`를 호출한다.
- `chunk_document()`는 `sentence_split()`으로 문장을 자른 뒤 슬라이딩 윈도우 방식으로 청크를 만든다.
- overlap을 두는 이유는 중요한 문장이 경계에서 끊겨도 다음 청크에 다시 포함되게 하기 위해서다.

**주요 파라미터/변수**
- `chunk_size`: 청크 하나에 넣을 문장 수다.
- `overlap`: 다음 청크와 몇 문장을 공유할지 정한다.
- `persist=False`: 이번 실습에서는 파일로 저장하지 않고 메모리에서만 확인한다.

**실제 소스 코드: chunk_document() — src/ingestion.py**
```python
def chunk_document(
    document: dict[str, str],
    chunk_size: int = DEFAULT_CHUNK_SIZE,
    overlap: int = DEFAULT_CHUNK_OVERLAP,
) -> list[dict[str, Any]]:
    sentences = sentence_split(document["text"])
    if not sentences:
        return []

    step = max(1, chunk_size - overlap)
    chunks: list[dict[str, Any]] = []
    for index in range(0, len(sentences), step):
        window = sentences[index : index + chunk_size]
        if not window:
            continue
        chunk_id = f"{document['doc_id']}_chunk_{len(chunks) + 1}"
        chunks.append(
            {
                "doc_id": document["doc_id"],
                "chunk_id": chunk_id,
                "text": " ".join(window),
                "source": document["source"],
            }
        )
        if index + chunk_size >= len(sentences):
            break
    return chunks
```

**코드 읽기 포인트**
- `sentences = sentence_split(document["text"])`: 먼저 문장을 기준으로 분할한다.
- `step = max(1, chunk_size - overlap)`: overlap을 반영한 실제 이동 간격이다.
- `window = sentences[index : index + chunk_size]`: 현재 청크에 들어갈 문장 묶음이다.
- `chunk_id = f"{document["doc_id"]}_chunk_{len(chunks)+1}"`: citation과 디버깅을 위해 안정적인 청크 ID를 붙인다.
- `if index + chunk_size >= len(sentences): break`: 마지막 청크를 만든 뒤 불필요한 빈 윈도우 생성을 막는다.

**결과 해석 가이드**
- 표에서 같은 `doc_id`가 여러 `chunk_id`로 쪼개져 보이면 청킹이 정상이다.
- 청크 텍스트가 너무 길면 검색 점수가 퍼지고, 너무 짧으면 문맥이 부족해진다.

**💡 면접 포인트**
- "문장 기반 청킹은 사람에게 설명하기 쉽고 trace를 읽기 편하지만, production에서는 토큰 길이와 문서 구조를 함께 고려해야 한다"고 말할 수 있다.


In [ ]:
from src.ingestion import ingest_documents

chunks = ingest_documents(persist=False)
chunk_frame = pd.DataFrame(chunks)
chunk_frame[['doc_id', 'chunk_id', 'source', 'text']].head(10)

## 검색 표현 방식: TF-IDF와 dense retrieval

아래 셀은 같은 corpus로 두 개의 retriever를 만든다. 하나는 현재 기본 경로인 TF-IDF 기반 `HybridRetriever`, 다른 하나는 `build_demo_index(backend='faiss')`를 통한 dense retrieval 경로다. 이 저장소에서는 optional dependency가 없으면 FAISS 경로가 다시 TF-IDF로 폴백할 수 있는데, 그 동작 자체도 실무적으로 중요하다. 환경에 따라 성능을 올리되, notebook은 항상 실행 가능해야 하기 때문이다.

**목적**
- 검색기를 어떻게 초기화하는지와 baseline의 기본 검색 표현을 이해한다.

**핵심 로직**
- `HybridRetriever.from_chunks()`는 청크 텍스트 전체에 TF-IDF vectorizer를 학습한다.
- 질문과 청크를 같은 벡터 공간에 올려 cosine similarity를 계산할 수 있게 된다.
- 이 notebook에서는 class name을 같이 출력해 dense backend가 실제로 무엇으로 연결됐는지도 확인한다.

**주요 파라미터/변수**
- `backend="tfidf"`: CPU 환경에서도 바로 동작하는 deterministic baseline이다.
- `backend="faiss"`: dense similarity 실험용 경로다. optional dependency가 없으면 폴백될 수 있다.
- `persist=False`: retriever 생성만 보고 인덱스 파일은 남기지 않는다.

**실제 소스 코드: HybridRetriever.from_chunks() — src/retriever.py**
```python
    @classmethod
    def from_chunks(cls, chunks: list[dict[str, Any]]) -> "HybridRetriever":
        vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
        matrix = vectorizer.fit_transform(chunk["text"] for chunk in chunks)
        return cls(chunks=chunks, vectorizer=vectorizer, matrix=matrix)
```

**코드 읽기 포인트**
- `TfidfVectorizer(ngram_range=(1, 2), stop_words="english")`: unigram과 bigram을 함께 보아 짧은 구문 일치까지 잡는다.
- `fit_transform(...)`: 청크 전체 corpus를 보고 vocabulary와 sparse matrix를 동시에 만든다.
- 이 시점의 출력은 임베딩 모델이 아니라 희소 벡터(sparse vector)다.
- Dense backend는 의미 유사성을 더 잘 잡을 수 있지만, 설치 비용과 실행 비용이 늘어난다.

**결과 해석 가이드**
- `class_name`이 `HybridRetriever`면 TF-IDF 경로, `FAISSRetriever`면 dense 경로가 실제로 활성화된 것이다.
- 교육용 데모에서는 어떤 backend가 선택됐는지 투명하게 보여주는 것이 중요하다.


In [ ]:
from src.ingestion import build_demo_index

tfidf_retriever = build_demo_index(persist=False, backend='tfidf')
faiss_retriever = build_demo_index(persist=False, backend='faiss')
retriever_summary = pd.DataFrame(
    [
        {'backend': 'tfidf', 'class_name': type(tfidf_retriever).__name__},
        {'backend': 'faiss', 'class_name': type(faiss_retriever).__name__},
    ]
)
retriever_summary

## 구현: 실제 검색 결과 읽기

이제 질의를 실제로 던져 상위 청크를 비교한다. TF-IDF는 단어 겹침에 강하고, dense retrieval은 의미적 유사성에 강하다는 설명을 많이 듣지만, 중요한 것은 결국 결과 표를 읽는 습관이다. 어떤 source가 상위로 올라왔고, 점수(score)가 어느 정도 차이 나는지를 직접 봐야 retrieval 품질을 설명할 수 있다.

TF-IDF의 직관도 여기서 잡아두자. 특정 문서에 자주 나오지만 corpus 전체에서는 흔하지 않은 단어일수록 가중치가 커진다. 그래서 질문의 핵심 단어가 어느 청크에 특이하게 등장하는지가 점수에 큰 영향을 준다.

**목적**
- 질의와 청크 간 유사도가 실제로 어떻게 순위로 나타나는지 확인한다.

**핵심 로직**
- `HybridRetriever.search()`는 질문을 같은 TF-IDF 공간으로 변환한 뒤 cosine similarity를 계산한다.
- 여기에 lexical overlap을 20% 섞어서, 완전히 의미 기반이 아니라 키워드 일치도 함께 반영한다.
- 정렬된 상위 `top_k` 청크를 가져와 표로 보여준다.

**주요 파라미터/변수**
- `retrieval_query`: 검색 품질을 읽기 위한 기준 질문이다.
- `top_k=4`: 가장 관련 높은 청크 4개만 본다.
- `score`: `cosine * 0.8 + lexical overlap * 0.2`로 합성된 최종 점수다.

**실제 소스 코드: HybridRetriever.search() — src/retriever.py**
```python
    def search(self, query: str, top_k: int = 5) -> list[dict[str, Any]]:
        if not self.chunks:
            return []

        query_vector = self.vectorizer.transform([query])
        cosine_scores = cosine_similarity(query_vector, self.matrix).flatten()
        lexical_scores = np.array(
            [overlap_ratio(content_tokens(query), content_tokens(chunk["text"])) for chunk in self.chunks]
        )
        combined_scores = (cosine_scores * 0.8) + (lexical_scores * 0.2)
        ranking = np.argsort(combined_scores)[::-1][:top_k]

        results: list[dict[str, Any]] = []
        for index in ranking:
            chunk = dict(self.chunks[index])
            chunk["score"] = round(float(combined_scores[index]), 4)
            results.append(chunk)
        return results
```

**코드 읽기 포인트**
- `query_vector = self.vectorizer.transform([query])`: 질문을 학습된 vocabulary 기준으로 벡터화한다.
- `cosine_similarity(query_vector, self.matrix)`: 질문과 각 청크의 방향 유사도를 계산한다.
- `lexical_scores = ... overlap_ratio(...)`: 핵심 토큰이 실제로 얼마나 겹치는지 별도 측정한다.
- `combined_scores = (cosine_scores * 0.8) + (lexical_scores * 0.2)`: 의미 유사도와 표면 단어 일치를 섞는다.
- `np.argsort(...)[::-1][:top_k]`: 점수 높은 순으로 뒤집어서 상위 k개만 남긴다.

**결과 해석 가이드**
- 대체로 `score`가 0.5 이상이면 매우 관련성 높은 청크, 0.3 안팎이면 읽어 보며 판단할 경계 구간, 그보다 낮으면 노이즈일 가능성이 크다.
- 두 backend 모두 같은 source를 상위에 올리면 retrieval이 안정적이라고 볼 수 있고, 결과가 크게 다르면 왜 다른지 텍스트를 직접 읽어야 한다.

**💡 면접 포인트**
- "검색기는 숫자만 보는 것이 아니라, 어떤 source가 왜 올라왔는지 표와 본문을 함께 읽어야 품질을 설명할 수 있다"고 말하면 좋다.


In [ ]:
retrieval_query = 'What are the goals of the workspace policy refresh?'
tfidf_results = pd.DataFrame(tfidf_retriever.search(retrieval_query, top_k=4))
faiss_results = pd.DataFrame(faiss_retriever.search(retrieval_query, top_k=4))

print('TF-IDF top results')
display(tfidf_results[['chunk_id', 'source', 'score', 'text']])
print('FAISS path top results')
display(faiss_results[['chunk_id', 'source', 'score', 'text']] if not faiss_results.empty else faiss_results)

## baseline 답변 생성(Simple QA)

이제 retrieval 결과를 바탕으로 baseline이 실제 답변을 어떻게 만드는지 본다. 이 단계에서 중요한 관찰은 "생성"이라고 해도 거대한 생성 모델을 부르는 것이 아니라, 이미 검색된 문장 중 관련도가 높은 문장을 골라 이어 붙이는 수준이라는 점이다. 그래서 빠르고 투명하지만, 복합 질문에서는 답변 구조가 단순해질 수밖에 없다.

**목적**
- 검색된 근거가 어떻게 draft answer로 바뀌는지 이해한다.

**핵심 로직**
- `build_baseline_answer()`는 질의와 가장 잘 맞는 문장 2개를 골라 이어 붙인다.
- `run_baseline_rag()`는 normalize → retrieve → synthesize 세 단계만 trace로 남긴다.
- 이 baseline에는 query classification, tool use, verifier, fallback이 없다.

**주요 파라미터/변수**
- `query`: 정규화된 질문 문자열
- `retrieved_docs`: 이미 검색된 상위 청크 목록
- `citations`: 답변과 함께 보여줄 상위 3개 근거 메타데이터

**실제 소스 코드: build_baseline_answer() — src/synthesizer.py**
```python
def build_baseline_answer(query: str, retrieved_docs: list[dict[str, Any]]) -> dict[str, Any]:
    ranked = _rank_sentences(query, retrieved_docs)
    evidence = [sentence for _, sentence, _ in ranked[:2]]
    citations = [
        {
            "doc_id": doc["doc_id"],
            "chunk_id": doc["chunk_id"],
            "source": doc["source"],
            "score": doc["score"],
        }
        for doc in retrieved_docs[:3]
    ]
    answer = " ".join(evidence) if evidence else "No relevant evidence was retrieved."
    return {"draft_answer": answer, "citations": citations}
```

**코드 읽기 포인트**
- `ranked = _rank_sentences(query, retrieved_docs)`: 먼저 청크 내부 문장을 다시 질의 기준으로 재정렬한다.
- `evidence = [sentence for _, sentence, _ in ranked[:2]]`: 상위 2개 문장만 답변 본문 후보로 쓴다.
- `citations = [...] for doc in retrieved_docs[:3]`: 답변에 포함한 문장과 완벽히 1:1 매칭되는 citation이 아니라, 우선 상위 청크 3개를 붙인다.
- `answer = " ".join(evidence)`: 문장 조합 중심이라 자연어 흐름은 제한적이다.
- 근거가 하나도 없으면 즉시 "No relevant evidence..."를 반환하므로 실패 모드가 비교적 투명하다.

**결과 해석 가이드**
- `final_status`가 항상 `answered`라는 점이 baseline의 가장 큰 한계다. 모르면 모른다고 말하지 못한다.
- `retrieved_sources`가 답변 내용과 잘 맞는지 함께 봐야, 문장 조합이 근거를 제대로 반영했는지 알 수 있다.


In [ ]:
from src.workflow import run_baseline_rag

baseline_result = run_baseline_rag(
    'What are the main goals of the workspace policy refresh?',
    retriever=tfidf_retriever,
)

pd.Series(
    {
        'final_status': baseline_result['final_status'],
        'final_answer': baseline_result['final_answer'],
        'retrieved_sources': [item['source'] for item in baseline_result['retrieved_docs']],
    }
)

## 실험: 쉬운 질문과 어려운 질문 비교

아래 셀은 같은 baseline을 쉬운 factual lookup과 날짜 계산이 필요한 질문에 각각 적용한다. 이 비교가 중요한 이유는, baseline이 단순 사실 조회에서는 그럭저럭 보이더라도, 계산이나 다단계 reasoning이 필요한 순간 급격히 취약해진다는 점을 눈으로 보여주기 때문이다.

**목적**
- baseline이 어떤 질문에서 잘 작동하고 어떤 질문에서 약한지 비교한다.

**핵심 로직**
- `run_baseline_rag()`는 normalize, retrieve, synthesize만 수행한다.
- 도구 호출이 없으므로 날짜 차이 계산도 결국 검색 문장을 그대로 붙인 답변에 머물 수 있다.
- 실험 결과를 DataFrame으로 정리해 question별 한계를 읽기 쉽게 만든다.

**주요 파라미터/변수**
- `experiment_questions`: baseline의 강점과 약점을 동시에 드러내는 질문 묶음
- `retrieved_sources`: 어떤 근거를 보고 답했는지 빠르게 점검하기 위한 컬럼

**실제 소스 코드: run_baseline_rag() — src/workflow.py**
```python
def run_baseline_rag(query: str, retriever: Any, top_k: int = DEFAULT_TOP_K) -> dict[str, Any]:
    start = time.perf_counter()
    normalize_start = time.perf_counter()
    normalized_query = normalize_text(query)
    normalize_latency = round(time.perf_counter() - normalize_start, 6)

    retrieval_start = time.perf_counter()
    retrieved_docs = retriever.search(normalized_query, top_k=top_k)
    retrieval_latency = round(time.perf_counter() - retrieval_start, 6)

    synthesis_start = time.perf_counter()
    synthesis = build_baseline_answer(normalized_query, retrieved_docs)
    synthesis_latency = round(time.perf_counter() - synthesis_start, 6)

    latency = time.perf_counter() - start
    return {
        "user_query": query,
        "normalized_query": normalized_query,
        "retrieved_docs": retrieved_docs,
        "draft_answer": synthesis["draft_answer"],
        "final_answer": synthesis["draft_answer"],
        "final_status": "answered",
        "citations": synthesis["citations"],
        "trace": [
            {
                "node": "normalize_query",
                "inputs": {"user_query": query},
                "outputs": {"normalized_query": normalized_query},
                "latency": normalize_latency,
                "payload": {
                    "inputs": {"user_query": query},
                    "outputs": {"normalized_query": normalized_query},
                    "latency": normalize_latency,
                },
            },
            {
                "node": "retrieve_docs",
                "inputs": {"normalized_query": normalized_query, "top_k": top_k},
                "outputs": {"results": len(retrieved_docs)},
                "latency": retrieval_latency,
                "payload": {
                    "inputs": {"normalized_query": normalized_query, "top_k": top_k},
                    "outputs": {"results": len(retrieved_docs)},
                    "latency": retrieval_latency,
                },
            },
            {
                "node": "synthesize_answer",
                "inputs": {"retrieved_doc_count": len(retrieved_docs)},
                "outputs": {"draft_answer": synthesis["draft_answer"]},
                "latency": synthesis_latency,
                "payload": {
                    "inputs": {"retrieved_doc_count": len(retrieved_docs)},
                    "outputs": {"draft_answer": synthesis["draft_answer"]},
                    "latency": synthesis_latency,
                },
            },
        ],
        "latency_seconds": round(latency, 4),
    }
```

**코드 읽기 포인트**
- `normalized_query = normalize_text(query)`: baseline도 최소한의 입력 정규화는 수행한다.
- `retrieved_docs = retriever.search(...)`: 검색 단계는 agent workflow와 동일한 retriever 인터페이스를 쓴다.
- `synthesis = build_baseline_answer(...)`: 생성은 별도 verifier 없이 곧바로 최종 답변이 된다.
- `trace`에 세 단계만 남기기 때문에 디버깅은 쉽지만, reasoning granularity는 낮다.
- `final_status`가 무조건 `answered`이므로 불충분 근거를 구분하지 못한다.

**결과 해석 가이드**
- 질문이 계산을 요구하는데도 답변이 단순 문장 인용에 그치면 baseline의 reasoning 한계가 드러난 것이다.
- `retrieved_sources`가 맞더라도 답변이 질문 형식에 정확히 대응하지 못할 수 있다. 이것이 retrieval과 answer synthesis를 분리해서 봐야 하는 이유다.

**💡 면접 포인트**
- "Baseline은 retrieval 품질을 보기엔 좋지만, answer formulation과 abstention policy를 평가하기에는 구조적으로 부족하다"고 설명할 수 있다.


In [ ]:
experiment_questions = [
    'When does the organization-wide rollout begin?',
    'How many days are in the pilot window?',
]
experiment_rows = []
for question in experiment_questions:
    result = run_baseline_rag(question, retriever=tfidf_retriever)
    experiment_rows.append(
        {
            'question': question,
            'final_status': result['final_status'],
            'retrieved_sources': ', '.join(sorted({doc['source'] for doc in result['retrieved_docs']})),
            'answer': result['final_answer'],
        }
    )

pd.DataFrame(experiment_rows)

## 결과 해석 가이드

여기까지의 결과는 "검색이 맞았는가"와 "답변 형식이 질문 요구에 맞았는가"를 분리해서 읽어야 한다. baseline은 retrieval이 잘 되더라도 답변이 계산형 질문을 직접 해결하지 못할 수 있고, 반대로 답변 문장은 그럴듯해도 근거 coverage가 충분한지 스스로 검증하지 않는다.

**목적**
- 다음 02번 notebook으로 넘어가기 전에 baseline의 한계를 구조적으로 정리한다.

**핵심 로직**
- TF-IDF baseline은 빠르고 재현 가능하다.
- Dense/FAISS 경로는 의미 유사성 회복 가능성을 보여준다.
- 하지만 둘 다 planning, tool orchestration, grounding verification은 제공하지 않는다.

**결과 해석 가이드**
- `strength`는 학습용 baseline의 장점, `limitation`은 agentic workflow가 보완해야 할 영역으로 읽으면 된다.
- 여기서 적힌 limitation이 바로 다음 notebook의 노드들로 이어진다: classify, plan, tools, verify, fallback.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {
            'backend': 'TF-IDF baseline',
            'strength': 'Fast, deterministic, easy to inspect',
            'limitation': 'No planning, tool use, or abstention logic',
        },
        {
            'backend': 'Optional FAISS path',
            'strength': 'Dense similarity can surface semantically close evidence',
            'limitation': 'Needs extra dependencies and more setup',
        },
    ]
)
analysis_frame

## 핵심 정리

이 실험을 통해 baseline RAG의 전체 흐름은 매우 투명하다는 점을 확인했다. `load_documents()`로 원문을 읽고, `chunk_document()`로 검색 단위를 만들고, `HybridRetriever.search()`로 관련 청크를 고른 뒤, `build_baseline_answer()`가 상위 문장을 이어 붙여 답을 만든다. 구조가 단순해서 디버깅과 설명은 쉽지만, 질문 유형을 분기하지 않고, 날짜 계산 같은 외부 도구도 쓰지 않으며, 답변이 실제 근거에 충분히 기반하는지 검증하지 않는다.

02번 notebook에서는 이 한계를 보완하기 위해 stateful workflow를 도입한다. 질의를 분류하고, 계획을 세우고, 필요하면 도구를 부르고, 마지막에 근거 검증(grounding verification)과 fallback을 수행한다.

**💡 면접 포인트**
- "Baseline RAG는 retrieval 실험과 디버깅에는 좋지만, 복합 질문과 안전한 abstain 정책에는 한계가 있다."
- "TF-IDF는 빠르고 설명 가능하며, dense/FAISS는 의미 유사성 회복 가능성이 있지만 환경 의존성이 늘어난다."
- "단순 RAG의 부족함이 분명해야 agentic workflow의 필요성을 설득할 수 있다."
